# Full 114M-Name SapBERT Contrastive Fine-Tuning

This notebook fine-tunes base SapBERT on the full reference name database without loading the 114M rows into memory at once. It streams the CSV, applies the same repository normalizer (`NFKC + lowercase + strip + whitespace collapse`), builds a compact per-CID synonym pool, and trains SapBERT with the same supervised contrastive batch pattern used by the smaller fine-tuning run.

The important design choice is that we do **not** materialize every synonym pair. For large synonym sets, all possible pairs grow combinatorially. Instead, the notebook samples a fixed number of normalized names per CID from the full database, then dynamically samples two names from each CID group during training. Increase `NAMES_PER_CID` if you want more positive-pair variety and have time/disk budget.


## Install Dependencies

Run this first in Colab. It avoids reinstalling PyTorch so Colab keeps the CUDA build that matches the runtime.


In [ ]:

import subprocess
import sys

packages = [
    'pandas==2.2.2',
    'numpy>=1.26.4,<2.1',
    'pyarrow>=15.0',
    'duckdb>=1.0',
    'tqdm>=4.66',
    'PyYAML>=6.0',
    'transformers>=4.40',
    'accelerate>=0.28',
    'sentencepiece>=0.2',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Installed notebook dependencies. If Colab asks for a restart after dependency changes, restart and continue from the next cell.')


## Mount Drive and Locate Repo


In [ ]:

from pathlib import Path
import os
import shutil
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


def find_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
        Path('/content/drive/MyDrive/chem_synonymizer'),
        Path('/content/chem_synonymizer'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer. Update find_repo_dir().')


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## Configuration

The defaults are chosen to match the smaller SapBERT contrastive fine-tune more closely: 128 CID groups per batch, two names sampled per CID group, supervised contrastive loss over the full in-batch similarity matrix, and 2,000 training steps. `NAMES_PER_CID = 4` gives each CID more pair variety without materializing all possible pairs.


In [ ]:

import gc
import json
import math
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

from chem_synonymizer.normalize import normalize_name
from chem_synonymizer.utils import save_json, set_seed

# Input data. Update only if your Drive path differs.
REFERENCE_CSV_PATH = REPO_DIR / 'data' / 'CID-Synonym-full.csv'
CID_COL = 'cid'
NAME_COL = 'name'

# Model and output locations.
BASE_MODEL_NAME = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'
RUN_NAME = 'sapbert_full_114m_contrastive_dynamic'
RESULTS_RUN_DIR = REPO_DIR / 'results' / RUN_NAME
RESULTS_ARTIFACTS_DIR = RESULTS_RUN_DIR / 'artifacts'
MODEL_OUTPUT_DIR = RESULTS_ARTIFACTS_DIR / 'model'
CHECKPOINT_ROOT = RESULTS_ARTIFACTS_DIR / 'checkpoints'

# Use local scratch for the heavy DuckDB and pair-building work.
SCRATCH_ROOT = Path('/mnt/local-scratch') if Path('/mnt/local-scratch').exists() else Path('/content')
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / RUN_NAME
WORK_ARTIFACTS_DIR = WORK_DIR / 'artifacts'
PREPARE_DB_PATH = WORK_ARTIFACTS_DIR / 'full114m_group_prep.duckdb'
TRAIN_GROUPS_PATH = WORK_ARTIFACTS_DIR / 'train_cid_groups.parquet'
GROUP_STATS_PATH = WORK_ARTIFACTS_DIR / 'group_stats.json'

# Data preparation.
SEED = 0
CPU_THREADS = max(1, os.cpu_count() or 8)
REFERENCE_CHUNKSIZE = 2_000_000
NAMES_PER_CID = 4
REBUILD_TRAINING_GROUPS = False
DELETE_PREPARE_DB_AFTER_PARQUET = True

# Training. One batch group contributes two texts sampled from the same CID, matching SynonymBatchFactory.contrastive_batch.
MAX_LENGTH = 96
POOLING = 'cls'
TRAIN_GROUP_BATCH_SIZE = 128
MAX_TRAIN_STEPS = 2_000
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
TEMPERATURE = 0.05
MIXED_PRECISION = 'bf16'  # A100-friendly. Use 'fp16' if bf16 is unavailable.
SAVE_EVERY_STEPS = 1000
KEEP_LAST_N_CHECKPOINTS = 2
OVERWRITE_MODEL = False
RESUME_FROM_CHECKPOINT = True

for path in [RESULTS_RUN_DIR, RESULTS_ARTIFACTS_DIR, CHECKPOINT_ROOT, WORK_ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not REFERENCE_CSV_PATH.exists():
    raise FileNotFoundError(REFERENCE_CSV_PATH)

set_seed(SEED)
print('Reference CSV:', REFERENCE_CSV_PATH)
print('Results:', RESULTS_RUN_DIR)
print('Scratch:', WORK_DIR, shutil.disk_usage(SCRATCH_ROOT))
print('CPU threads:', CPU_THREADS)


## Helpers


In [ ]:

def normalize_series(values: pd.Series) -> pd.Series:
    # Vectorized equivalent of chem_synonymizer.normalize.normalize_name.
    return (
        values.fillna('')
        .astype('string')
        .str.normalize('NFKC')
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )


def q(path: Path) -> str:
    return str(path).replace("'", "''")


def result_artifact_path(work_path: Path) -> Path:
    return RESULTS_ARTIFACTS_DIR / work_path.name


def restore_artifact(work_path: Path) -> None:
    result_path = result_artifact_path(work_path)
    if work_path.exists() or not result_path.exists():
        return
    work_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Restoring {result_path.name} to scratch...')
    shutil.copy2(result_path, work_path)


def backup_artifact(work_path: Path) -> None:
    if not work_path.exists():
        return
    result_path = result_artifact_path(work_path)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    if result_path.exists() and result_path.stat().st_size == work_path.stat().st_size:
        print('Drive artifact already backed up:', result_path.name)
        return
    if result_path.exists():
        result_path.unlink()
    print(f'Backing up {work_path.name} to Drive results...')
    shutil.copy2(work_path, result_path)
    if result_path.stat().st_size != work_path.stat().st_size:
        raise OSError(f'Backup size mismatch for {work_path.name}')


def save_config() -> None:
    payload = {
        'run_name': RUN_NAME,
        'reference_csv_path': str(REFERENCE_CSV_PATH),
        'base_model_name': BASE_MODEL_NAME,
        'normalization': 'NFKC + lowercase + strip + whitespace collapse',
        'cid_col': CID_COL,
        'name_col': NAME_COL,
        'seed': SEED,
        'cpu_threads': CPU_THREADS,
        'reference_chunksize': REFERENCE_CHUNKSIZE,
        'names_per_cid': NAMES_PER_CID,
        'max_length': MAX_LENGTH,
        'pooling': POOLING,
        'train_group_batch_size': TRAIN_GROUP_BATCH_SIZE,
        'max_train_steps': MAX_TRAIN_STEPS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
        'max_grad_norm': MAX_GRAD_NORM,
        'temperature': TEMPERATURE,
        'mixed_precision': MIXED_PRECISION,
        'model_output_dir': str(MODEL_OUTPUT_DIR),
    }
    save_json(payload, RESULTS_RUN_DIR / 'full114m_finetune_config.json')


save_config()


## Build Full-Database CID Groups

This stage streams `CID-Synonym-full.csv`, normalizes names, removes duplicate `(cid, normalized_name)` pairs, keeps CIDs with at least two normalized names, samples `NAMES_PER_CID` names per CID, and writes one row per CID group to Parquet.

During training, each batch samples two names from each CID group dynamically. This matches the smaller `SynonymBatchFactory.contrastive_batch` pattern more closely than the previous fixed-pair version.


In [ ]:

def prepare_training_groups() -> dict:
    restore_artifact(TRAIN_GROUPS_PATH)
    restore_artifact(GROUP_STATS_PATH)
    if TRAIN_GROUPS_PATH.exists() and GROUP_STATS_PATH.exists() and not REBUILD_TRAINING_GROUPS:
        with GROUP_STATS_PATH.open('r', encoding='utf-8') as f:
            stats = json.load(f)
        print('Using cached training CID groups:', TRAIN_GROUPS_PATH)
        print(json.dumps(stats, indent=2)[:3000])
        return stats

    for path in [TRAIN_GROUPS_PATH, GROUP_STATS_PATH, PREPARE_DB_PATH]:
        if path.exists() and path.is_file():
            path.unlink()

    con = duckdb.connect(str(PREPARE_DB_PATH))
    con.execute(f'PRAGMA threads={int(CPU_THREADS)}')
    con.execute('PRAGMA preserve_insertion_order=false')
    con.execute('DROP TABLE IF EXISTS reference_pairs')
    con.execute('CREATE TABLE reference_pairs (cid VARCHAR, normalized_name VARCHAR)')

    total_rows = 0
    kept_rows = 0
    chunk_i = 0
    t0 = time.perf_counter()
    reader = pd.read_csv(
        REFERENCE_CSV_PATH,
        usecols=[CID_COL, NAME_COL],
        dtype={CID_COL: 'string', NAME_COL: 'string'},
        chunksize=REFERENCE_CHUNKSIZE,
        on_bad_lines='skip',
    )
    for chunk in tqdm(reader, desc='Streaming and normalizing reference CSV'):
        chunk_i += 1
        total_rows += len(chunk)
        out = pd.DataFrame({
            'cid': chunk[CID_COL].astype('string'),
            'normalized_name': normalize_series(chunk[NAME_COL]),
        })
        out = out.dropna(subset=['cid'])
        out = out[out['normalized_name'].str.len() > 0]
        out = out.drop_duplicates(['cid', 'normalized_name'])
        kept_rows += len(out)
        con.register('chunk_df', out)
        con.execute('INSERT INTO reference_pairs SELECT CAST(cid AS VARCHAR), CAST(normalized_name AS VARCHAR) FROM chunk_df')
        con.unregister('chunk_df')
        del chunk, out
        if chunk_i % 5 == 0:
            gc.collect()

    print('Creating global deduplicated reference table...')
    con.execute('DROP TABLE IF EXISTS reference_dedup')
    con.execute('''
        CREATE TABLE reference_dedup AS
        SELECT DISTINCT cid, normalized_name
        FROM reference_pairs
        WHERE cid IS NOT NULL AND normalized_name IS NOT NULL AND length(normalized_name) > 0
    ''')
    con.execute('DROP TABLE reference_pairs')

    print('Counting trainable CIDs...')
    con.execute('DROP TABLE IF EXISTS cid_counts')
    con.execute('''
        CREATE TABLE cid_counts AS
        SELECT cid, COUNT(*) AS n_names
        FROM reference_dedup
        GROUP BY cid
        HAVING COUNT(*) >= 2
    ''')

    print(f'Sampling up to {NAMES_PER_CID} names per CID...')
    con.execute('DROP TABLE IF EXISTS sampled_names')
    con.execute(f'''
        CREATE TABLE sampled_names AS
        SELECT cid, normalized_name, rn
        FROM (
            SELECT
                d.cid,
                d.normalized_name,
                ROW_NUMBER() OVER (
                    PARTITION BY d.cid
                    ORDER BY hash(d.normalized_name || '|' || CAST({int(SEED)} AS VARCHAR))
                ) AS rn
            FROM reference_dedup d
            INNER JOIN cid_counts c USING (cid)
        )
        WHERE rn <= {int(NAMES_PER_CID)}
    ''')

    print('Writing sampled CID groups...')
    con.execute(f'''
        COPY (
            SELECT
                a.cid,
                list(a.normalized_name ORDER BY a.rn) AS names,
                COUNT(*) AS n_names
            FROM sampled_names a
            GROUP BY a.cid
            HAVING COUNT(*) >= 2
            ORDER BY hash(a.cid || '|group|' || CAST({int(SEED)} AS VARCHAR))
        ) TO '{q(TRAIN_GROUPS_PATH)}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    ''')

    stats = {
        'total_csv_rows_seen': int(total_rows),
        'chunk_dedup_rows_inserted': int(kept_rows),
        'unique_cid_name_pairs': int(con.execute('SELECT COUNT(*) FROM reference_dedup').fetchone()[0]),
        'trainable_cids': int(con.execute('SELECT COUNT(*) FROM cid_counts').fetchone()[0]),
        'sampled_name_rows': int(con.execute('SELECT COUNT(*) FROM sampled_names').fetchone()[0]),
        'train_groups': int(con.execute(f"SELECT COUNT(*) FROM read_parquet('{q(TRAIN_GROUPS_PATH)}')").fetchone()[0]),
        'names_per_cid': int(NAMES_PER_CID),
        'batch_sampling': 'dynamic_two_names_per_cid_group_supervised_contrastive',
        'elapsed_sec': float(time.perf_counter() - t0),
    }
    save_json(stats, GROUP_STATS_PATH)
    con.close()

    if DELETE_PREPARE_DB_AFTER_PARQUET and PREPARE_DB_PATH.exists():
        PREPARE_DB_PATH.unlink()

    backup_artifact(TRAIN_GROUPS_PATH)
    backup_artifact(GROUP_STATS_PATH)
    print(json.dumps(stats, indent=2))
    return stats


group_stats = prepare_training_groups()


## Fine-Tune SapBERT with Supervised Contrastive Loss

If a final model already exists, the cell skips training unless `OVERWRITE_MODEL = True`. Checkpoints are written to Drive so a Colab disconnect does not destroy the whole run.


In [ ]:

import torch
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from chem_synonymizer.embed import pool_outputs


def latest_checkpoint() -> Path | None:
    checkpoints = []
    for path in CHECKPOINT_ROOT.glob('checkpoint-*'):
        try:
            step = int(path.name.split('-')[-1])
        except ValueError:
            continue
        if (path / 'config.json').exists():
            checkpoints.append((step, path))
    if not checkpoints:
        return None
    return sorted(checkpoints)[-1][1]


def cleanup_old_checkpoints() -> None:
    checkpoints = []
    for path in CHECKPOINT_ROOT.glob('checkpoint-*'):
        try:
            step = int(path.name.split('-')[-1])
        except ValueError:
            continue
        checkpoints.append((step, path))
    for _, path in sorted(checkpoints)[:-int(KEEP_LAST_N_CHECKPOINTS)]:
        shutil.rmtree(path, ignore_errors=True)


def supervised_contrastive_loss(embeddings: torch.Tensor, labels: torch.Tensor, temperature: float) -> torch.Tensor:
    labels = labels.to(embeddings.device)
    sim = torch.matmul(embeddings, embeddings.T) / float(temperature)
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    eye = torch.eye(labels.numel(), dtype=torch.bool, device=embeddings.device)
    positive_mask = labels[:, None].eq(labels[None, :]) & ~eye
    logits_mask = ~eye
    exp_logits = torch.exp(sim) * logits_mask.float()
    log_prob = sim - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))
    positives_per_anchor = positive_mask.sum(dim=1)
    valid = positives_per_anchor > 0
    mean_log_prob = (positive_mask.float() * log_prob).sum(dim=1) / positives_per_anchor.clamp_min(1)
    return -mean_log_prob[valid].mean()


def encode_train_batch(texts: list[str], tokenizer, model, device: str) -> torch.Tensor:
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=int(MAX_LENGTH),
        return_tensors='pt',
    ).to(device)
    if MIXED_PRECISION == 'bf16':
        amp_dtype = torch.bfloat16
        amp_enabled = device.startswith('cuda')
    elif MIXED_PRECISION == 'fp16':
        amp_dtype = torch.float16
        amp_enabled = device.startswith('cuda')
    else:
        amp_dtype = torch.float32
        amp_enabled = False
    with torch.autocast(device_type='cuda' if device.startswith('cuda') else 'cpu', dtype=amp_dtype, enabled=amp_enabled):
        out = model(**enc)
        emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    del enc, out
    return emb


def contrastive_group_batch_iterator(parquet_path: Path, batch_groups: int, seed: int, skip_batches: int = 0):
    pf = pq.ParquetFile(parquet_path)
    rng = np.random.default_rng(int(seed) + int(skip_batches))
    seen_batches = 0
    while True:
        for batch in pf.iter_batches(batch_size=int(batch_groups), columns=['cid', 'names']):
            if seen_batches < skip_batches:
                seen_batches += 1
                continue
            df = batch.to_pandas()
            df = df.dropna(subset=['cid', 'names'])
            if df.empty:
                seen_batches += 1
                continue
            texts = []
            labels = []
            for group_i, names in enumerate(df['names'].tolist()):
                names = [str(name) for name in list(names) if str(name)]
                if len(names) < 2:
                    continue
                a_i, p_i = rng.choice(len(names), size=2, replace=False)
                texts.extend([names[int(a_i)], names[int(p_i)]])
                labels.extend([int(group_i), int(group_i)])
            if not texts:
                seen_batches += 1
                continue
            labels = torch.tensor(labels, dtype=torch.long)
            seen_batches += 1
            yield texts, labels


def save_training_checkpoint(model, tokenizer, step: int, history: list[dict]) -> None:
    ckpt_dir = CHECKPOINT_ROOT / f'checkpoint-{int(step):06d}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    save_json({'step': int(step), 'history_tail': history[-10:]}, ckpt_dir / 'training_state.json')
    pd.DataFrame(history).to_csv(RESULTS_RUN_DIR / 'full114m_finetune_history.csv', index=False)
    cleanup_old_checkpoints()


device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device.startswith('cuda'):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
print('Training device:', device)

if (MODEL_OUTPUT_DIR / 'config.json').exists() and not OVERWRITE_MODEL:
    print('Final model already exists, skipping training:', MODEL_OUTPUT_DIR)
else:
    start_step = 0
    model_source = BASE_MODEL_NAME
    ckpt = latest_checkpoint() if RESUME_FROM_CHECKPOINT else None
    if ckpt is not None and not OVERWRITE_MODEL:
        model_source = str(ckpt)
        try:
            start_step = int(ckpt.name.split('-')[-1])
        except ValueError:
            start_step = 0
        print('Resuming model weights from checkpoint:', ckpt)
    else:
        print('Starting from base model:', BASE_MODEL_NAME)

    tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=True)
    model = AutoModel.from_pretrained(model_source).to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=float(LEARNING_RATE), weight_decay=float(WEIGHT_DECAY))
    total_steps = int(MAX_TRAIN_STEPS)
    warmup_steps = int(total_steps * float(WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    # If resuming weights only, advance the scheduler approximately to the resumed step.
    for _ in range(start_step):
        scheduler.step()

    history = []
    running_loss = None
    iterator = contrastive_group_batch_iterator(TRAIN_GROUPS_PATH, TRAIN_GROUP_BATCH_SIZE, seed=SEED, skip_batches=start_step)
    progress = tqdm(range(start_step, total_steps), desc='Fine-tuning SapBERT on full reference CID groups')
    t0 = time.perf_counter()

    for step in progress:
        texts, labels = next(iterator)
        optimizer.zero_grad(set_to_none=True)
        embeddings = encode_train_batch(texts, tokenizer, model, device)
        loss = supervised_contrastive_loss(embeddings, labels, TEMPERATURE)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(MAX_GRAD_NORM))
        optimizer.step()
        scheduler.step()

        loss_value = float(loss.detach().cpu())
        running_loss = loss_value if running_loss is None else (0.95 * running_loss + 0.05 * loss_value)
        progress.set_postfix(loss=f'{running_loss:.4f}')

        if ((step + 1) % 100 == 0) or (step + 1 == total_steps):
            history.append({
                'step': int(step + 1),
                'loss': loss_value,
                'running_loss': float(running_loss),
                'elapsed_sec': float(time.perf_counter() - t0),
            })

        if ((step + 1) % int(SAVE_EVERY_STEPS) == 0) or (step + 1 == total_steps):
            save_training_checkpoint(model, tokenizer, step + 1, history)

        del texts, labels, embeddings, loss
        if device.startswith('cuda') and (step + 1) % 25 == 0:
            torch.cuda.empty_cache()

    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_OUTPUT_DIR)
    tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
    pd.DataFrame(history).to_csv(RESULTS_RUN_DIR / 'full114m_finetune_history.csv', index=False)
    save_json(
        {
            'run_name': RUN_NAME,
            'base_model_name': BASE_MODEL_NAME,
            'output_dir': str(MODEL_OUTPUT_DIR),
            'train_groups_path': str(TRAIN_GROUPS_PATH),
            'group_stats': group_stats,
            'max_train_steps': int(MAX_TRAIN_STEPS),
            'final_running_loss': history[-1]['running_loss'] if history else None,
            'elapsed_sec': float(time.perf_counter() - t0),
        },
        RESULTS_RUN_DIR / 'full114m_finetune_metrics.json',
    )
    print('Saved final model:', MODEL_OUTPUT_DIR)


## Quick Pair-Alignment Sanity Check

This is not the final retrieval evaluation. It only checks whether dynamically sampled same-CID name pairs have higher cosine similarity than random in-batch negatives on a small sample of the training-group file.


In [ ]:

def quick_pair_alignment_check(sample_pairs: int = 4096) -> dict:
    import torch
    from transformers import AutoModel, AutoTokenizer

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_DIR, use_fast=True)
    model = AutoModel.from_pretrained(MODEL_OUTPUT_DIR).to(device).eval()

    groups = pq.read_table(TRAIN_GROUPS_PATH, columns=['names']).slice(0, int(sample_pairs)).to_pandas()
    rng = np.random.default_rng(SEED)
    anchors = []
    positives = []
    for names in groups['names'].tolist():
        names = [str(name) for name in list(names) if str(name)]
        if len(names) < 2:
            continue
        a_i, p_i = rng.choice(len(names), size=2, replace=False)
        anchors.append(names[int(a_i)])
        positives.append(names[int(p_i)])
    texts = anchors + positives
    embeddings = []
    for start in tqdm(range(0, len(texts), 512), desc='Encoding sanity-check pairs'):
        batch = texts[start:start + 512]
        with torch.inference_mode():
            emb = encode_train_batch(batch, tokenizer, model, device)
        embeddings.append(emb.detach().cpu())
    emb = torch.cat(embeddings, dim=0)
    n = len(anchors)
    anchor_emb = emb[:n]
    positive_emb = emb[n:]
    pos_sim = (anchor_emb * positive_emb).sum(dim=1).numpy()
    neg_sim = (anchor_emb * positive_emb[torch.randperm(n)]).sum(dim=1).numpy()
    payload = {
        'sample_pairs': int(n),
        'positive_cosine_mean': float(pos_sim.mean()),
        'random_negative_cosine_mean': float(neg_sim.mean()),
        'positive_gt_random_fraction': float((pos_sim > neg_sim).mean()),
    }
    save_json(payload, RESULTS_RUN_DIR / 'quick_pair_alignment_check.json')
    print(json.dumps(payload, indent=2))
    return payload


if (MODEL_OUTPUT_DIR / 'config.json').exists():
    sanity = quick_pair_alignment_check()
else:
    print('No final model found yet; run the fine-tuning cell first.')


## Next Step

The fine-tuned checkpoint is saved at:

```text
results/sapbert_full_114m_contrastive_dynamic/artifacts/model
```

To evaluate it with the external-lab FAISS notebook, point that notebook's `MODEL_DIR` at this model directory and use a separate results folder so the old `sapbert_contrastive` results remain intact.
